# 02 — NLP Pipeline
**Goal:** Transform raw MD&A text into structured, analysis-ready features.

 **Steps:**
 1. Load extracted MD&A from Stage 1
 2. Sentence tokenisation
 3. Boilerplate removal (legal disclaimers, forward-looking statement noise)
 4. TF-IDF vectorisation to surface distinctive risk language
 5. Top keyword extraction per section
 **Input:** `outputs/jpm_mda_2025Q2.json`  
 **Output:** `outputs/jpm_sentences_clean.json` -->

In [3]:
import json
import pathlib

input_path = pathlib.Path("../outputs/jpm_mda_2025Q2.json")

with open(input_path, "r", encoding = "utf-8") as f:
    data = json.load(f)
mda_text = data["mda_text"] 

print(f"Company: {data['company']}")
print(f"Period: {data['period']}")
print(f"MD&A length: {len(mda_text):,} characters")

Company: JPMorgan Chase & Co.
Period: 2025-06-30
MD&A length: 597,528 characters


In [6]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(mda_text)

print(f"Total sentences: {len(sentences)}")
print("\nSample sentences:")
for s in sentences[10:15]:
    print(f"- {s[:120]}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ozoha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ozoha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Total sentences: 2305

Sample sentences:
- Under the J.P. Morgan and Chase brands, the Firm serves millions of customers, predominantly in the U.S., and many of th
- JPMorganChase’s principal bank subsidiary is JPMorgan Chase Bank, National Association (“JPMorgan Chase Bank, N.A.”), a 
- Morgan Securities”), a U.S. broker-dealer.
- The bank and non-bank subsidiaries of JPMorganChase operate nationally as well as through overseas branches and subsidia
- The Firm’s principal operating subsidiaries outside the U.S. are J.P. Morgan Securities plc and J.P. Morgan SE (“JPMSE”)


In [7]:
import re

BOILERPLATE_PATTERNS = [
    r"forward.looking statement",
    r"private securities litigation reform act",
    r"actual results.{0,50}differ materially",
    r"refer to.{0,80}for (a )?(further |more )?discussion",
    r"see note \d+",
    r"refer to note \d+",
    r"for more information",
    r"incorporated herein by reference",
    r"^\s*\d+\s*$",               # page numbers
    r"jpmorgan chase & co\.",     # repetitive firm name boilerplate
    r"form 10-q",
    r"this (quarterly|annual) report",
]

def is_boilerplate(sentence):
    s = sentence.lower()
    for pattern in BOILERPLATE_PATTERNS:
        if re.search(pattern, s):
            return True
    if len(sentence.split()) < 8:   # too short to carry signal
        return True
    return False

clean_sentences = [s for s in sentences if not is_boilerplate(s)]

removed = len(sentences) - len(clean_sentences)
print(f"Original:  {len(sentences):,} sentences")
print(f"Removed:   {removed:,} boilerplate sentences ({removed/len(sentences)*100:.1f}%)")
print(f"Remaining: {len(clean_sentences):,} sentences")

print("\nSample clean sentences:")
for s in clean_sentences[20:25]:
    print(f"  — {s[:150]}")

Original:  2,305 sentences
Removed:   279 boilerplate sentences (12.1%)
Remaining: 2,026 sentences

Sample clean sentences:
  — • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income ("NII") was $23.2 billion, up 2%, driven by higher Markets net inte
  — These factors were predominantly offset by the impact of lower rates and deposit margin compression.
  — NII excluding Markets was $22.8 billion, down 1%.
  — – Noninterest revenue ("NIR") was $21.7 billion, down 21%, primarily reflecting the absence of the $7.9 billion net gain related to Visa shares record
  — Excluding this net gain, NIR would have been up 11% driven by higher Markets noninterest revenue, lower net investment securities losses in Treasury a


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# TF-IDF across all sentences — surfaces terms distinctive to THIS filing
vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words="english",
    ngram_range=(1, 2),      # unigrams + bigrams (e.g. "credit risk", "net interest")
    min_df=2,                # term must appear in at least 2 sentences
)

tfidf_matrix = vectorizer.fit_transform(clean_sentences)
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(feature_names):,} terms")
print(f"Matrix shape: {tfidf_matrix.shape}")

Vocabulary size: 500 terms
Matrix shape: (2026, 500)


In [9]:
import numpy as np

# Sum TF-IDF scores across all sentences — highest = most distinctive terms
scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
top_indices = scores.argsort()[::-1][:40]

print("Top 40 most distinctive terms in JPM Q2 2025 MD&A:\n")
for i, idx in enumerate(top_indices):
    print(f"  {i+1:>2}. {feature_names[idx]:<35} {scores[idx]:.3f}")

Top 40 most distinctive terms in JPM Q2 2025 MD&A:

   1. firm                                113.969
   2. credit                              68.187
   3. loans                               65.188
   4. 2025                                61.706
   5. 2024                                57.653
   6. 30                                  56.502
   7. net                                 55.802
   8. billion                             53.045
   9. june                                52.724
  10. june 30                             51.962
  11. risk                                51.543
  12. securities                          51.220
  13. value                               51.086
  14. assets                              46.423
  15. related                             45.943
  16. 30 2025                             45.739
  17. income                              44.953
  18. losses                              43.713
  19. capital                             43.383
  20. fair      

In [10]:
# Add domain-specific stop words to remove numeric/date noise
custom_stop_words = [
    "30", "31", "2024", "2025", "june", "december", "january",
    "billion", "million", "firm", "certain", "related", "including",
    "following", "based", "used", "using", "also", "may", "one",
    "two", "three", "increase", "decrease", "increased", "decreased",
    "compared", "prior", "period", "quarter", "year", "ended",
    "page", "table", "note", "refer", "information", "form"
]

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

vectorizer2 = TfidfVectorizer(
    max_features=500,
    stop_words=all_stop_words,
    ngram_range=(1, 2),
    min_df=3,
)

tfidf_matrix2 = vectorizer2.fit_transform(clean_sentences)
feature_names2 = vectorizer2.get_feature_names_out()

scores2 = np.asarray(tfidf_matrix2.sum(axis=0)).flatten()
top_indices2 = scores2.argsort()[::-1][:40]

print("Top 40 distinctive financial terms — cleaned:\n")
for i, idx in enumerate(top_indices2):
    print(f"  {i+1:>2}. {feature_names2[idx]:<40} {scores2[idx]:.3f}")

Top 40 distinctive financial terms — cleaned:

   1. credit                                   70.987
   2. loans                                    68.479
   3. net                                      60.278
   4. value                                    53.748
   5. securities                               53.642
   6. risk                                     53.252
   7. assets                                   49.611
   8. income                                   47.943
   9. losses                                   46.675
  10. capital                                  46.079
  11. fair                                     42.673
  12. months                                   41.310
  13. fair value                               40.621
  14. management                               35.624
  15. revenue                                  34.365
  16. financial                                34.228
  17. total                                    33.585
  18. market                       

In [11]:
# Define risk phrase categories relevant to credit/market risk analysis
RISK_CATEGORIES = {
    "credit_risk": [
        "credit loss", "allowance", "charge-off", "nonperforming",
        "net charge", "loan loss", "default", "delinquency",
        "credit quality", "impairment", "provisioning"
    ],
    "market_risk": [
        "value at risk", "var", "market risk", "interest rate risk",
        "volatility", "fair value", "hedging", "derivatives",
        "net interest", "spread"
    ],
    "macro_signals": [
        "economic outlook", "recession", "inflation", "federal reserve",
        "rate hike", "unemployment", "gdp", "macroeconomic",
        "geopolitical", "uncertainty", "tariff", "trade"
    ],
    "liquidity_risk": [
        "liquidity", "funding", "deposit", "borrowing",
        "cash flow", "stress test", "hqla", "lcr"
    ]
}

def flag_risk_sentences(sentences, categories):
    flagged = []
    for sentence in sentences:
        s_lower = sentence.lower()
        matched_categories = []
        matched_phrases = []
        for category, phrases in categories.items():
            hits = [p for p in phrases if p in s_lower]
            if hits:
                matched_categories.append(category)
                matched_phrases.extend(hits)
        if matched_categories:
            flagged.append({
                "sentence": sentence,
                "categories": matched_categories,
                "phrases": matched_phrases
            })
    return flagged

flagged = flag_risk_sentences(clean_sentences, RISK_CATEGORIES)

print(f"Risk-flagged sentences: {len(flagged):,} of {len(clean_sentences):,} clean sentences")
print(f"Coverage: {len(flagged)/len(clean_sentences)*100:.1f}%\n")

print("Sample flagged sentences:\n")
for item in flagged[:5]:
    print(f"  Categories: {item['categories']}")
    print(f"  Phrases:    {item['phrases']}")
    print(f"  Text: {item['sentence'][:200]}")
    print()

Risk-flagged sentences: 827 of 2,026 clean sentences
Coverage: 40.8%

Sample flagged sentences:

  Categories: ['credit_risk', 'market_risk']
  Phrases:    ['credit loss', 'net interest']
  Text: Financial performance of JPMorganChase (unaudited) As of or for the period ended, (in millions, except per share data and ratios) Three months ended June 30, Six months ended June 30, 2025 2024 Change

  Categories: ['market_risk']
  Phrases:    ['net interest']
  Text: (c) NII and NIR refer to net interest income and noninterest revenue, respectively.

  Categories: ['market_risk', 'liquidity_risk']
  Phrases:    ['net interest', 'deposit']
  Text: • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income ("NII") was $23.2 billion, up 2%, driven by higher Markets net interest income, higher wholesale deposit balances, hi

  Categories: ['liquidity_risk']
  Phrases:    ['deposit']
  Text: These factors were predominantly offset by the impact of lower rates and deposit 

In [12]:
# Filter out flagged sentences that are too short to be analytical
MIN_WORDS = 15

flagged_clean = [
    item for item in flagged
    if len(item["sentence"].split()) >= MIN_WORDS
]

print(f"After minimum word filter: {len(flagged_clean):,} sentences")

# Preview quality check
print("\nTop flagged sentences by category:\n")
for category in RISK_CATEGORIES.keys():
    cat_sentences = [f for f in flagged_clean if category in f["categories"]]
    print(f"[{category.upper()}] — {len(cat_sentences)} sentences")
    if cat_sentences:
        print(f"  Example: {cat_sentences[0]['sentence'][:200]}")
    print()

After minimum word filter: 764 sentences

Top flagged sentences by category:

[CREDIT_RISK] — 198 sentences
  Example: Financial performance of JPMorganChase (unaudited) As of or for the period ended, (in millions, except per share data and ratios) Three months ended June 30, Six months ended June 30, 2025 2024 Change

[MARKET_RISK] — 410 sentences
  Example: Financial performance of JPMorganChase (unaudited) As of or for the period ended, (in millions, except per share data and ratios) Three months ended June 30, Six months ended June 30, 2025 2024 Change

[MACRO_SIGNALS] — 72 sentences
  Example: The net addition to the allowance for credit losses of $439 million, primarily in wholesale, was driven by the impact of net lending activity, largely offset by the impact of changes in the Firm’s wei

[LIQUIDITY_RISK] — 236 sentences
  Example: • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income ("NII") was $23.2 billion, up 2%, driven by higher Markets net in

In [13]:
import json
from datetime import datetime

output = {
    "company": data["company"],
    "ticker": data["ticker"],
    "period": data["period"],
    "processed_at": datetime.now().isoformat(),
    "stats": {
        "total_sentences": len(sentences),
        "clean_sentences": len(clean_sentences),
        "risk_flagged": len(flagged_clean),
        "coverage_pct": round(len(flagged_clean) / len(clean_sentences) * 100, 1)
    },
    "top_tfidf_terms": [
        {"term": feature_names2[idx], "score": round(float(scores2[idx]), 3)}
        for idx in top_indices2[:30]
    ],
    "flagged_sentences": flagged_clean
}

out_path = pathlib.Path("../outputs/jpm_nlp_pipeline_2025Q2.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved: {out_path.resolve()}")
print(f"\nStage 2 complete.")
print(f"  Clean sentences:  {output['stats']['clean_sentences']:,}")
print(f"  Risk flagged:     {output['stats']['risk_flagged']:,}")
print(f"  Coverage:         {output['stats']['coverage_pct']}%")

Saved: C:\Users\Ozoha\nlp_finance_sentiment\outputs\jpm_nlp_pipeline_2025Q2.json

Stage 2 complete.
  Clean sentences:  2,026
  Risk flagged:     764
  Coverage:         37.7%


In [14]:
TABLE_NOISE_PATTERNS = [
    r"unaudited",
    r"in millions",
    r"per share",
    r"three months ended",
    r"six months ended",
    r"nine months ended",
    r"as of or for",
    r"^\s*\(",              # lines starting with parenthetical
    r"refer to page",
    r"see accompanying notes",
]

def is_table_noise(sentence):
    s = sentence.lower()
    for pattern in TABLE_NOISE_PATTERNS:
        if re.search(pattern, s):
            return True
    return False

flagged_final = [
    item for item in flagged_clean
    if not is_table_noise(item["sentence"])
]

print(f"After table noise filter: {len(flagged_final):,} sentences")
print(f"Removed: {len(flagged_clean) - len(flagged_final)} table/header sentences\n")

print("Quality check — one example per category:\n")
for category in RISK_CATEGORIES.keys():
    cat_sentences = [f for f in flagged_final if category in f["categories"]]
    print(f"[{category.upper()}] — {len(cat_sentences)} sentences")
    if cat_sentences:
        print(f"  {cat_sentences[0]['sentence'][:220]}")
    print()

After table noise filter: 548 sentences
Removed: 216 table/header sentences

Quality check — one example per category:

[CREDIT_RISK] — 131 sentences
  The net addition to the allowance for credit losses of $439 million, primarily in wholesale, was driven by the impact of net lending activity, largely offset by the impact of changes in the Firm’s weighted-average macroe

[MARKET_RISK] — 282 sentences
  • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income ("NII") was $23.2 billion, up 2%, driven by higher Markets net interest income, higher wholesale deposit balances, higher revolving balan

[MACRO_SIGNALS] — 61 sentences
  The net addition to the allowance for credit losses of $439 million, primarily in wholesale, was driven by the impact of net lending activity, largely offset by the impact of changes in the Firm’s weighted-average macroe

[LIQUIDITY_RISK] — 167 sentences
  • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income

In [15]:
output_final = {
    "company": data["company"],
    "ticker": data["ticker"],
    "period": data["period"],
    "processed_at": datetime.now().isoformat(),
    "stats": {
        "total_sentences": len(sentences),
        "clean_sentences": len(clean_sentences),
        "risk_flagged": len(flagged_final),
        "coverage_pct": round(len(flagged_final) / len(clean_sentences) * 100, 1)
    },
    "top_tfidf_terms": [
        {"term": feature_names2[idx], "score": round(float(scores2[idx]), 3)}
        for idx in top_indices2[:30]
    ],
    "flagged_sentences": flagged_final
}

out_path = pathlib.Path("../outputs/jpm_nlp_pipeline_2025Q2.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output_final, f, ensure_ascii=False, indent=2)

print(f"Saved: {out_path.name}")
print(f"  Total sentences:  {output_final['stats']['total_sentences']:,}")
print(f"  Clean sentences:  {output_final['stats']['clean_sentences']:,}")
print(f"  Risk flagged:     {output_final['stats']['risk_flagged']:,}")
print(f"  Coverage:         {output_final['stats']['coverage_pct']}%")
print("\nStage 2 complete.")

Saved: jpm_nlp_pipeline_2025Q2.json
  Total sentences:  2,305
  Clean sentences:  2,026
  Risk flagged:     548
  Coverage:         27.0%

Stage 2 complete.
